# 12 · A Foundry model as a tool, not a brain

## Goal

Deploy a cheap extraction model on Foundry and reach it from the agent as a
prompt tool — not as the agent's reasoning model, which is fixed by
`copilot.yaml`'s `model:` block and comes only from the built-in list.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
settings = load_settings()
settings.require("AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZURE_LOCATION")


## Concept

**Finding #6 is the most important correction in this curriculum.** BYOM
from Foundry is scoped to prompt tools on the GHCP harness — it is never
the agent's own reasoning model. If you came in wanting to "connect the
agent to a model deployed on Foundry," what actually happens is: the
Foundry deployment becomes a callable tool (a prompt tool, a workflow AI
action, or a Function-wrapped endpoint) that the built-in reasoning model
can invoke, the same way it would invoke any other tool. The agent's brain
still comes from `copilot.yaml`'s fixed model list.

This reframing is also the setup for `22`'s cost story: since the
reasoning model is fixed per agent (finding #4), cheap-model economics
happen either via tool calls like this one, or via connected agents each
pinned to their own model — never by swapping the spine agent's own model
mid-conversation.


## Build


In [ ]:
import subprocess, json
deploy = subprocess.run([
    "az", "deployment", "group", "create",
    "--resource-group", settings.get("AZURE_RESOURCE_GROUP"),
    "--template-file", "../infra/bicep/modules/foundry.bicep",
    "--parameters", "namePrefix=crd-dev", f"location={settings.get('AZURE_LOCATION')}",
], capture_output=True, text=True)
outputs = json.loads(deploy.stdout)["properties"]["outputs"] if deploy.returncode == 0 else {}
print(outputs)


### Wire it in as a prompt tool


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")
tools_dir = workspace / "tools"
tools_dir.mkdir(exist_ok=True)

prompt_tool = {
    "id": "extract-renewal-clauses",
    "type": "prompt-tool",
    "displayName": "Extract renewal clauses (Foundry, gpt-4o-mini)",
    "description": "Extracts supplier name, notice period, and uplift percentage from free-text renewal notices. Cheap, fast, deterministic-ish extraction — not for drafting or judgement calls.",
    "foundryEndpoint": outputs.get("projectEndpoint", {}).get("value"),
    "deploymentName": outputs.get("extractionDeploymentName", {}).get("value", "extraction-mini"),
    "promptTemplate": "Extract supplier name, notice period, and uplift percentage as JSON from: {{input}}",
}
(tools_dir / "extract-renewal-clauses.yaml").write_text(yaml.dump(prompt_tool, sort_keys=False))

from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


### State plainly what did NOT change


In [ ]:
copilot_yaml = (workspace / "copilot.yaml").read_text()
assert "model:" in copilot_yaml and "gpt-5-mini" in copilot_yaml, "reasoning model is unchanged — Foundry only added a tool"
print("Confirmed: copilot.yaml's model: block is untouched. The Foundry deployment is reachable only as a tool call.")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
suite = run_suite(client, cases=load_golden(tags=["foundry"]) + load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("12", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="Foundry deploy (separate Azure billing) + tool-call verification (Copilot Credits)")


## Teardown


In [ ]:
print("No teardown — the extraction tool persists; 13 adds an MCP toolset alongside it.")
